# Скоринг лидов: обучение модели

## Задача и метрика
Оценить вероятность целевого действия в течение 5 дней после назначения обращения. Метрика —
Daily Average Precision: AP считается внутри каждой даты назначения и усредняется по дням
(эквивалент MAP с группой = день). Абсолютный уровень score не важен — важен порядок внутри дня.

## Идея решения
Табличные признаки дополняем инженерными признаками из `events.csv` (подробнее в `event_features.py`). События заметно поднимают
качество, но их легко испортить утечкой: в истории присутствуют события *после* назначения,
попадающие в окно таргета. Поэтому в фичи берём только `event_ts < assignment_ts`. Основная
модель — CatBoost с целевой `MAP` (group_id = день) и строго временной валидацией. Финальный
прогноз — гетерогенный ансамбль CatBoost + LightGBM + XGBoost: внутридневные перцентильные
ранги каждой модели усредняются (для Daily AP важен только порядок внутри дня, ранги снимают
разницу калибровок между библиотеками).

Пайплайн воспроизводим и считает признаки на лету: `EventFeatureBuilder` из `event_features.py`
строит 74 event-признака, мёрж с таблицей и обучение — здесь. Гиперпараметры трёх библиотек
подтягиваются из `cb/lgb/xgb_best_params.json`

## Ход решения
Каждую гипотезу проверял ablation'ом на 3 временных фолдах (не на одном сплите) и дешёвым
префильтром на ортогональность — R² предсказуемости признака из остальных.

| Шаг | Daily AP | Комментарий |
|---|---|---|
| только табличные | 0.545 | точка отсчёта |
| + event-фичи (74) | 0.697 | +0.15, основной вклад |
| тюнинг CatBoost (Optuna, 3-fold CV) | 0.6977 | depth=4 + сильная регуляризация |
| ансамбль по 5 сидам | 0.6987 | снятие дисперсии сида |
| + LightGBM + XGBoost, rank-бленд | 0.7033 (CV) | ошибки библиотек не совпадают; прирост на всех фолдах |
| − мультисид cb внутри бленда | без изменений | бленд сам гасит дисперсию (CV 0.6970 vs 0.6968); в ~3 раза быстрее |
| тюнинг LightGBM/XGBoost (Optuna, те же фолды) | 0.7044 (CV) | одиночки 0.6996/0.6994; обе сошлись к мелким деревьям и lr≈0.02, как cb; бленд +0.002 на всех фолдах |

Отклонено по ablation (прироста нет):
- 25 отдельных биграмм переходов — шум; оставлены только 4 агрегата (`ev_bg_*`).
- Суточный профиль / каденс / воронка — избыточны с recency/activity (R² ≈ 0.99).
- Сегментная нормировка по `region`/`car_segment` — R² ≈ 1, полностью избыточна.
- Прунинг по важности — стабильно хуже: малозначимый хвост даёт коллективный сигнал.
- Ручная обработка NaN (isna-флаги, счётчик пропусков, `nan_mode=Max`) — нативный `Min`-режим
  CatBoost уже извлекает сигнал пропуска; явные флаги только добавляют шум.
- Расширение бленда (logreg / RF / ExtraTrees / HistGBM) — все кандидаты сильно слабее тройки
  (0.58–0.67 vs 0.69), их шум перевешивает разнообразие; каждое добавление ухудшает CV.
- Тюнинг весов бленда (leave-one-fold-out) — 0.6971 vs 0.6970 у равных: при сопоставимых по силе
  компонентах равные веса оптимальны.
- Мета-модель (LR-стэкинг поверх рангов) — 0.6930 < 0.6970: на 13.7k строк / 16 дней мета-уровень
  переобучается, простое усреднение сильнее.

Набор источников (события + табличные агрегаты) близок к потолку — дальнейший прирост требует
нового источника данных, а не новых формул над теми же.

In [15]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

from event_features import EventFeatureBuilder, LEAD_KEY, ASSIGNMENT_TS, EVENT_TS

RANDOM_SEED = 42
DATA_DIR = Path("data")
PARAM_DIR = Path("params")
TARGET = "target"
DATE_COL = "assignment_date"
DROP_COLUMNS = [LEAD_KEY, "user_id", ASSIGNMENT_TS, DATE_COL, TARGET]
VALID_DAYS_FRACTION = 0.2

## Гиперпараметры
Все три библиотеки — из `cb/lgb/xgb_best_params.json`, без файла — дефолты.

In [16]:
DEFAULT_CB_PARAMS = dict(learning_rate=0.03, depth=6, l2_leaf_reg=3.0)
DEFAULT_LGB_PARAMS = dict(learning_rate=0.05, num_leaves=31, min_child_samples=50,
                          reg_lambda=5.0, colsample_bytree=0.8, subsample=0.8)
DEFAULT_XGB_PARAMS = dict(learning_rate=0.05, max_depth=5, min_child_weight=20,
                          reg_lambda=5.0, colsample_bytree=0.8, subsample=0.8)


def load_tuned_params(file_name, default_params):
    path = PARAM_DIR / Path(file_name)
    if not path.exists():
        print(f"{file_name} не найден -> дефолтные параметры")
        return default_params, None
    tuned = json.loads(path.read_text(encoding="utf-8"))
    print(f"{file_name}: CV Daily AP = {tuned['cv_daily_ap']}, folds = {tuned['fold_scores']}")
    return tuned["params"], tuned.get("final_iterations")


CB_PARAMS, CB_FINAL_ITERATIONS = load_tuned_params("cb_best_params.json", DEFAULT_CB_PARAMS)
LGB_PARAMS, _ = load_tuned_params("lgb_best_params.json", DEFAULT_LGB_PARAMS)
XGB_PARAMS, _ = load_tuned_params("xgb_best_params.json", DEFAULT_XGB_PARAMS)

cb_best_params.json: CV Daily AP = 0.6977, folds = [0.686, 0.7155, 0.6916]
lgb_best_params.json: CV Daily AP = 0.6996, folds = [0.6913, 0.7078, 0.6998]
xgb_best_params.json: CV Daily AP = 0.6994, folds = [0.689, 0.7095, 0.6996]


## Метрика

In [17]:
def average_precision(y_true: np.ndarray, y_score: np.ndarray) -> float:
    order = np.argsort(-y_score, kind="mergesort")
    y = y_true[order]
    n_positive = y.sum()
    if n_positive == 0:
        return np.nan
    cumulative_tp = np.cumsum(y)
    rank = np.arange(1, len(y) + 1)
    return float(np.sum((cumulative_tp / rank) * y) / n_positive)


def daily_average_precision(y_true, y_score, dates) -> float:
    frame = pd.DataFrame({"y": y_true, "s": y_score, "d": dates})
    scores = [average_precision(g["y"].to_numpy(), g["s"].to_numpy()) for _, g in frame.groupby("d")]
    scores = [s for s in scores if not np.isnan(s)]
    return float(np.mean(scores)) if scores else np.nan

## Загрузка данных

In [18]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=[ASSIGNMENT_TS])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=[ASSIGNMENT_TS])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=[EVENT_TS])

train[DATE_COL] = pd.to_datetime(train[DATE_COL]).dt.date
test[DATE_COL] = pd.to_datetime(test[DATE_COL]).dt.date
print("train:", train.shape, "| test:", test.shape, "| events:", events.shape)

train: (13694, 119) | test: (4306, 118) | events: (254705, 7)


## Event-фичи на лету и мёрж
`EventFeatureBuilder` фильтрует `event_ts < assignment_ts` — утечки нет по построению.

In [19]:
builder = EventFeatureBuilder()
key_columns = [LEAD_KEY, ASSIGNMENT_TS, "item_price_log"]

train_event_features = builder.build(events, train[key_columns])
test_event_features = builder.build(events, test[key_columns])

train_full = train.merge(train_event_features, on=LEAD_KEY, how="left")
test_full = test.merge(test_event_features, on=LEAD_KEY, how="left")

event_feature_names = [c for c in train_event_features.columns if c != LEAD_KEY]
print("event-фич добавлено:", len(event_feature_names))
print("train_full:", train_full.shape, "| test_full:", test_full.shape)
assert (train_full[LEAD_KEY].to_numpy() == train[LEAD_KEY].to_numpy()).all()

event-фич добавлено: 74
train_full: (13694, 193) | test_full: (4306, 192)


## Определение признаков
Категориальные (нечисловые) заполняем `"NA"` — CatBoost требует строки без пропусков; числовые
NaN он обрабатывает сам.

In [20]:
feature_columns = [c for c in train_full.columns if c not in DROP_COLUMNS and c in test_full.columns]
categorical_columns = [c for c in feature_columns if not pd.api.types.is_numeric_dtype(train_full[c])]
numeric_columns = [c for c in feature_columns if c not in categorical_columns]

for df in (train_full, test_full):
    for col in categorical_columns:
        df[col] = df[col].astype("object").where(df[col].notna(), "NA").astype(str)

print("всего признаков:", len(feature_columns))
print("категориальных:", len(categorical_columns), categorical_columns)
print("числовых:", len(numeric_columns), "(из них event:", len(event_feature_names), ")")

всего признаков: 188
категориальных: 7 ['lead_source', 'call_center', 'region', 'car_segment', 'lead_channel', 'user_tenure_bucket', 'price_bucket']
числовых: 181 (из них event: 74 )


## Временной сплит для валидации
Последние ~20% дней train — в hold-out (повторяет реальный сплит train→test).

In [21]:
ordered_dates = sorted(train_full[DATE_COL].unique())
cutoff = ordered_dates[int(len(ordered_dates) * (1 - VALID_DAYS_FRACTION))]

train_part = train_full[train_full[DATE_COL] < cutoff].copy()
valid_part = train_full[train_full[DATE_COL] >= cutoff].copy()
print("cutoff date:", cutoff)
print("train_part:", train_part.shape, "| valid_part:", valid_part.shape)
print("target rate train/valid:", round(train_part[TARGET].mean(), 3), "/", round(valid_part[TARGET].mean(), 3))

cutoff date: 2026-04-19
train_part: (10272, 193) | valid_part: (3422, 193)
target rate train/valid: 0.208 / 0.205


## Сборка Pool и обучение
`MAP` требует сгруппированных по `group_id` строк — Pool сортируем по дате.

In [22]:
def make_pool(df, feature_cols, cat_cols, with_label=True):
    ordered = df.sort_values(DATE_COL, kind="mergesort").reset_index(drop=True)
    group_id = pd.factorize(ordered[DATE_COL])[0]
    label = ordered[TARGET].to_numpy() if with_label else None
    pool = Pool(
        data=ordered[feature_cols],
        label=label,
        cat_features=cat_cols,
        group_id=group_id,
    )
    return pool, ordered


def fit_and_score(feature_cols, cat_cols, iterations=2000, verbose=0):
    train_pool, _ = make_pool(train_part, feature_cols, cat_cols)
    valid_pool, valid_ordered = make_pool(valid_part, feature_cols, cat_cols)

    model = CatBoostClassifier(
        iterations=iterations,
        loss_function="Logloss",
        eval_metric="MAP",
        random_seed=RANDOM_SEED,
        early_stopping_rounds=150,
        verbose=verbose,
        **CB_PARAMS,
    )
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    predictions = model.predict_proba(valid_ordered[feature_cols])[:, 1]
    y_valid = valid_ordered[TARGET].to_numpy()
    dates_valid = valid_ordered[DATE_COL].to_numpy()
    daily = daily_average_precision(y_valid, predictions, dates_valid)
    global_ap = average_precision_score(y_valid, predictions)
    return model, daily, global_ap


def valid_predictions_for_seed(feature_cols, cat_cols, seed, iterations=2000):
    train_pool, _ = make_pool(train_part, feature_cols, cat_cols)
    valid_pool, valid_ordered = make_pool(valid_part, feature_cols, cat_cols)
    model = CatBoostClassifier(
        iterations=iterations,
        loss_function="Logloss",
        eval_metric="MAP",
        random_seed=seed,
        early_stopping_rounds=150,
        verbose=0,
        **CB_PARAMS,
    )
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
    predictions = model.predict_proba(valid_ordered[feature_cols])[:, 1]
    return predictions, valid_ordered

## Ablation: вклад event-фич
Табличные vs табличные + event на одном сплите.

In [23]:
tabular_only = [c for c in feature_columns if c not in event_feature_names]
tabular_cat = [c for c in tabular_only if c in categorical_columns]

model_base, daily_base, ap_base = fit_and_score(tabular_only, tabular_cat)
model_full, daily_full, ap_full = fit_and_score(feature_columns, categorical_columns)

print("=== Валидационный Daily AP ===")
print(f"tabular-only     : {daily_base:.4f}   (global AP {ap_base:.4f})")
print(f"tabular + events : {daily_full:.4f}   (global AP {ap_full:.4f})")
print(f"прирост от events: {daily_full - daily_base:+.4f}")

=== Валидационный Daily AP ===
tabular-only     : 0.5488   (global AP 0.5440)
tabular + events : 0.6906   (global AP 0.6863)
прирост от events: +0.1419


In [24]:
cb_valid, valid_ordered = valid_predictions_for_seed(feature_columns, categorical_columns, RANDOM_SEED)
y_valid = valid_ordered[TARGET].to_numpy()
dates_valid = valid_ordered[DATE_COL].to_numpy()
cb_daily = daily_average_precision(y_valid, cb_valid, dates_valid)
print(f"catboost : Daily AP {cb_daily:.4f}")

catboost : Daily AP 0.6906


## Гетерогенный ансамбль: + LightGBM + XGBoost
Ошибки разных библиотек не совпадают, поэтому бленд сильнее любой одиночной модели. Смешиваем
внутридневные перцентильные ранги (Daily AP зависит только от порядка внутри дня). Проверено
на 3-фолд CV: cb 0.6977 / lgb 0.6996 / xgb 0.6994 → rank-бленд трёх 0.7044, прирост на всех фолдах.

In [25]:
LGB_BASE = dict(n_estimators=3000, subsample_freq=1, random_state=RANDOM_SEED, verbose=-1)
XGB_BASE = dict(
    n_estimators=3000, tree_method="hist", enable_categorical=True,
    eval_metric="aucpr", random_state=RANDOM_SEED, verbosity=0,
)

train_full_cat = train_full.copy()
test_full_cat = test_full.copy()
for col in categorical_columns:
    shared_dtype = pd.CategoricalDtype(sorted(set(train_full[col]) | set(test_full[col])))
    train_full_cat[col] = train_full_cat[col].astype(shared_dtype)
    test_full_cat[col] = test_full_cat[col].astype(shared_dtype)

train_part_cat = train_full_cat.loc[train_part.index]
valid_part_cat = train_full_cat.loc[valid_part.index]
valid_cat_ordered = valid_part_cat.sort_values(DATE_COL, kind="mergesort").reset_index(drop=True)
assert (valid_cat_ordered[LEAD_KEY].to_numpy() == valid_ordered[LEAD_KEY].to_numpy()).all()


def day_ranks(scores, dates):
    return pd.Series(scores).groupby(pd.Series(dates)).rank(pct=True).to_numpy()


lgb_model = LGBMClassifier(**LGB_BASE, **LGB_PARAMS)
lgb_model.fit(
    train_part_cat[feature_columns], train_part_cat[TARGET],
    eval_set=[(valid_cat_ordered[feature_columns], valid_cat_ordered[TARGET])],
    eval_metric="average_precision", callbacks=[early_stopping(150, verbose=False)],
)
lgb_valid = lgb_model.predict_proba(valid_cat_ordered[feature_columns])[:, 1]
lgb_best_iter = lgb_model.best_iteration_

xgb_model = XGBClassifier(early_stopping_rounds=150, **XGB_BASE, **XGB_PARAMS)
xgb_model.fit(
    train_part_cat[feature_columns], train_part_cat[TARGET],
    eval_set=[(valid_cat_ordered[feature_columns], valid_cat_ordered[TARGET])], verbose=False,
)
xgb_valid = xgb_model.predict_proba(valid_cat_ordered[feature_columns])[:, 1]
xgb_best_iter = xgb_model.best_iteration

for name, preds in [("lightgbm", lgb_valid), ("xgboost", xgb_valid)]:
    print(f"{name:9s}: Daily AP {daily_average_precision(y_valid, preds, dates_valid):.4f}")

blend_valid = np.mean(
    [day_ranks(p, dates_valid) for p in (cb_valid, lgb_valid, xgb_valid)], axis=0
)
blend_daily = daily_average_precision(y_valid, blend_valid, dates_valid)
print(f"rank-бленд cb+lgb+xgb: {blend_daily:.4f}  ({blend_daily - cb_daily:+.4f} к catboost)")
print(f"best_iter: lgb {lgb_best_iter}, xgb {xgb_best_iter}")

lightgbm : Daily AP 0.7023
xgboost  : Daily AP 0.7027
rank-бленд cb+lgb+xgb: 0.7053  (+0.0147 к catboost)
best_iter: lgb 1127, xgb 1419


## Важность признаков

In [26]:
importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model_full.get_feature_importance(),
}).sort_values("importance", ascending=False)
importance["is_event"] = importance["feature"].isin(event_feature_names)

event_share = importance.loc[importance["is_event"], "importance"].sum()
print(f"суммарная важность event-фич: {event_share:.1f}% от общей")
importance.head(25)

суммарная важность event-фич: 42.4% от общей


,feature,importance,is_event
165,ev_ctx_nunique,7.851966,True
174,ev_price_std,4.341061,True
45,seller_page_views_14d,4.260508,False
46,seller_page_views_30d,4.141392,False
44,seller_page_views_7d,4.115794,False
178,ev_price_diff_last,2.544178,True
162,ev_recency_hot,2.133073,True
0,lead_source,2.072191,False
123,ev_favorite_3d,2.044195,True
127,ev_hot_3d,1.794851,True


## Финал: гетерогенный ансамбль на всём train и предсказание test
По одной модели от библиотеки: CatBoost — итерации из тюнинга, LightGBM/XGBoost — из ранней
остановки на hold-out. Итоговый score — среднее внутридневных рангов трёх моделей
(дни — `assignment_date` теста).

In [27]:
cb_iterations = CB_FINAL_ITERATIONS or (model_full.get_best_iteration() + 1)
print("итераций для финального catboost:", cb_iterations)

full_pool, _ = make_pool(train_full, feature_columns, categorical_columns)
cb_final = CatBoostClassifier(
    iterations=cb_iterations,
    loss_function="Logloss",
    random_seed=RANDOM_SEED,
    verbose=0,
    **CB_PARAMS,
)
cb_final.fit(full_pool)
cb_test_scores = cb_final.predict_proba(test_full[feature_columns])[:, 1]

lgb_final = LGBMClassifier(**{**LGB_BASE, **LGB_PARAMS, "n_estimators": lgb_best_iter})
lgb_final.fit(train_full_cat[feature_columns], train_full_cat[TARGET])
lgb_test_scores = lgb_final.predict_proba(test_full_cat[feature_columns])[:, 1]

xgb_final = XGBClassifier(**{**XGB_BASE, **XGB_PARAMS, "n_estimators": xgb_best_iter})
xgb_final.fit(train_full_cat[feature_columns], train_full_cat[TARGET])
xgb_test_scores = xgb_final.predict_proba(test_full_cat[feature_columns])[:, 1]

test_dates = test_full[DATE_COL].to_numpy()
test_scores = np.mean(
    [day_ranks(s, test_dates) for s in (cb_test_scores, lgb_test_scores, xgb_test_scores)], axis=0
)
submission = pd.DataFrame({"lead_id": test_full[LEAD_KEY], "score": test_scores})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv: {submission.shape}")
print(submission.head())
print("score stats:", submission["score"].describe()[["min", "mean", "max"]].round(4).to_dict())

итераций для финального catboost: 992
submission.csv: (4306, 2)
                 lead_id     score
0  lead_97e409eb8f8c8246  0.244750
1  lead_55310edb4489f9e9  0.665893
2  lead_e7f653a2c6a7eee8  0.979381
3  lead_22f8e1cfc487ac20  0.106529
4  lead_48b638b839abfac3  0.554097
score stats: {'min': 0.0012, 'mean': 0.5006, 'max': 1.0}
